# SNR-PER-010 추론 및 성능평가

## 시험 개요
- 시험 대상: MOT20 훈련 영상 4개 시퀀스(MOT20-01 / 02 / 03 / 05, 8,931프레임)의 다중객체추적 정확도
- 판정 기준: TrackEval CLEAR 지표의 MOTA를 IoU 0.1 조건에서 시퀀스별로 산출, 4개 값의 **산술평균 80% 이상**이면 합격
- 시험 방법: 아래 2종 중 선택

| 방법 | 내용 | 소요 시간 | 기본 |
|---|---|---|---|
| A. 빠른 평가 | 사전 계산된 추적 결과(`tracking_results/`)의 무결성 확인 후 TrackEval 평가, 모델 미실행 | 수 분 | 예 |
| B. 전체 추론 | 영상 프레임 8,931장을 Grounding DINO·OSNet·ByteTrack·학습 기반 필터로 처음부터 추론 후 평가 | 약 10시간 (A40 GPU) | 아니오 |

## 방법 A의 입력
- 추적 결과(`tracking_results/`): 방법 B로 생성한 MOT 형식 txt 4개, SHA-256 목록은 `MANIFEST.json`
- 확인 항목: 추적 결과 변조 여부(SHA-256), 정답(GT) 파일의 공식 배포본 일치 여부(SHA-256)
- 확인 후 처리: 점수 재계산

## 폴더 구성 (노트북 위치: `notebooks/`)

| 폴더 | 내용 |
|---|---|
| `code/pipeline/` | 실행 진입점과 단계별 스크립트 (`run_inference.py`, `evaluate_tracking_results.py`, `download_dataset.py`, `verify_inputs.py` ...) |
| `code/src`, `code/scripts`, `code/configs`, `code/external/TrackEval` | 추적·검출 라이브러리, 처리 스크립트, 설정, 평가 도구 |
| `config/INFERENCE_PROFILE.json` | 고정된 최종 추적·후처리 운용점 |
| `models/` | 사전학습 모델 60개(Grounding DINO, OSNet, fold별 보정·필터·연결 모델)와 SHA-256 목록 |
| `tracking_results/` | 사전 계산된 추적 결과(MOT 형식 txt 4개)와 `MANIFEST.json` |
| `data/` | 데이터셋 내려받기 위치와 정보(`DATASET_MANIFEST.json`) |
| `results/` | 시험 결과 |

## 1. 시험 폴더와 설정
- 시험 폴더 탐색: 노트북 위치에서 상위 폴더로 올라가며 `code/pipeline/run_inference.py`가 있는 폴더 지정
- 결과 저장 위치: `results/<RUN_NAME>/`
- 시험 방법 선택: `RUN_FULL_INFERENCE` (False: 방법 A, True: 방법 B)

In [ ]:
import json
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

from IPython.display import Image, clear_output, display

here = Path.cwd().resolve()
package = next((folder for folder in [here, *here.parents] if (folder / 'code/pipeline/run_inference.py').is_file()), None)
assert package is not None, 'snr_per_010 폴더 안의 노트북 실행 필요'
pipeline = package / 'code/pipeline'
workspace = package / 'workspace'            # 전체 추론의 실행 폴더·로그 (git 제외)
data_root = package / 'data'                 # MOT20 내려받기 위치
RUN_NAME = 'evaluation-' + datetime.now(ZoneInfo('Asia/Seoul')).strftime('%Y%m%d-%H%M%S')
RUN_FULL_INFERENCE = False                   # True: 방법 B (약 10시간), False: 방법 A (기본)

print('시험 폴더 :', package)
print('Python    :', sys.executable)
print('결과 폴더 :', package / 'results' / RUN_NAME)

## 2. 시험 환경 확인
- 방법 A: CPU만으로 실행
- 출력 항목: 평가 도구(TrackEval) 사용 패키지 버전, GPU 유무
- 방법 B: `code/pipeline/check_reference_runtime.py`가 CUDA·패키지 버전 추가 검사

In [ ]:
import importlib.metadata as metadata
for name in ['numpy', 'scipy', 'pandas', 'matplotlib']:
    try:
        print(f'{name:12s}', metadata.version(name))
    except metadata.PackageNotFoundError:
        print(f'{name:12s}', '미설치 -> pip install ' + name)
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
print('GPU         :', gpu.stdout.strip() if gpu.returncode == 0 else '없음 (방법 A는 불필요)')

## 3. 데이터셋(정답) 내려받기
- 내려받기 경로: `https://motchallenge.net/data/MOT20.zip` (MOTChallenge 공식 배포본, 약 5 GB)
- 방법 A: 정답 파일(`gt/gt.txt`)과 `seqinfo.ini`만 사용, 영상 프레임 불필요
  - 내려받기 대상: 압축 파일 내 8개 파일(수 MB), HTTP 범위 요청(`--gt-only`)
  - 저장 위치: `data/MOT20/train/`
- 방법 B: 전체 데이터셋(프레임 8,931장) 내려받기
- 이미 받은 파일: 재내려받기 생략

In [ ]:
subprocess.run([sys.executable, str(pipeline / 'download_dataset.py'), '--workspace', str(workspace), '--data-root', str(data_root), '--gt-only'], check=True)

## 4. 모델·정답 검증
- 실행 스크립트: `verify_inputs.py`
- 검증 항목
  1. Grounding DINO 가중치: 64 MiB 조각 결합 후 SHA-256 검증 (`models/WEIGHT_PARTS.json`)
  2. 모델 파일 60개: 크기·SHA-256 검증 (`models/MODEL_MANIFEST.json`)
  3. 4개 시퀀스: 길이(`seqinfo.ini`)와 정답 파일 SHA-256 검증
- 불일치 시: 오류로 중단

In [ ]:
subprocess.run([sys.executable, str(pipeline / 'verify_inputs.py'), '--workspace', str(workspace), '--data-root', str(data_root), '--gt-only'], check=True)

## 5. 방법 A — 사전 계산된 추적 결과 평가
- 실행 스크립트: `evaluate_tracking_results.py`
- 처리 순서
  1. `tracking_results/*.txt` 4개의 SHA-256을 `MANIFEST.json` 기록과 대조
  2. 정답 파일의 SHA-256 대조
  3. TrackEval(CLEAR·Identity)로 IoU 0.1·0.5 점수 계산 (추론·학습 없음)
  4. 지표 표, 그래프, 출처 기록(`evaluation-provenance.json`)을 `results/<RUN_NAME>/`에 저장
  5. 산술평균 MOTA 80% 이상 여부 판정

In [ ]:
result = package / 'results' / RUN_NAME
subprocess.run([sys.executable, str(pipeline / 'evaluate_tracking_results.py'), '--workspace', str(workspace), '--data-root', str(data_root), '--output', str(result)], check=True)

## 6. 결과 확인
- 표시 항목: 시퀀스별 MOTA·FP·FN·IDSW·GT 개수, 4개 시퀀스 산술평균, 판정
- 비교 항목: 기록된 기준값(`tracking_results/MANIFEST.json`)과의 차이

In [ ]:
import pandas as pd

metrics = json.loads((result / 'evaluation/metrics.json').read_text(encoding='utf-8'))
primary = next(item for item in metrics['evaluations'] if item['match_iou'] == 0.1)
display(pd.DataFrame(primary['folds']).round(6))
provenance = json.loads((result / 'evaluation-provenance.json').read_text(encoding='utf-8'))
print(f"4개 시퀀스 평균 MOTA (IoU 0.1): {primary['macro_mota_percent']:.6f}%")
print(f"통합 MOTA (GT 개수 가중)      : {primary['pooled_mota_percent']:.6f}%")
print(f"기록된 기준값과의 차이        : {provenance['difference_from_reference_percentage_points']:+.12f} %p")
print('판정                          :', 'PASS (80% 이상)' if primary['macro_mota_percent'] >= 80 else 'FAIL')
display(Image(filename=str(result / 'charts/mota_comparison.png')))
display(Image(filename=str(result / 'charts/error_breakdown.png')))

## 7. (선택) 방법 B — 전체 추론
- 실행 조건: `RUN_FULL_INFERENCE = True` 설정 후 이 셀부터 실행
- 필요 자원: NVIDIA GPU, 공식 데이터셋 전체(프레임 8,931장), A40 GPU 기준 약 10시간
- 처리 순서
  1. 공식 데이터셋 전체 내려받기 (`download_dataset.py`)
  2. 모델·프레임·정답 검증 (`verify_inputs.py`)
  3. 추론 (`run_inference.py`): Grounding DINO 사람 후보 생성 → OSNet 특징 추출 → fold별 보정 모델 → ByteTrack → 학습 기반 추적 필터·연결 → 보간·누락 구간 보정
  4. TrackEval 평가와 산술평균 MOTA 80% 판정
- 재실행: 같은 실행 이름이면 완료된 단계 이후부터 이어서 처리
- 진행 상황 표시 주기: 15초

In [ ]:
if RUN_FULL_INFERENCE:
    full_run = 'full-' + RUN_NAME
    subprocess.run([sys.executable, str(pipeline / 'download_dataset.py'), '--workspace', str(workspace), '--data-root', str(data_root)], check=True)
    log_path = workspace / 'logs' / f'{full_run}.log'
    log_path.parent.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, str(pipeline / 'run_inference.py'), '--workspace', str(workspace), '--data-root', str(data_root), '--run-name', full_run]
    with log_path.open('w') as stream:
        process = subprocess.Popen(command, stdout=stream, stderr=subprocess.STDOUT)
        while process.poll() is None:
            clear_output(wait=True)
            for name in ['progress.json', 'remaining-progress.json', 'remaining-detector-progress.json']:
                path = workspace / 'runs' / full_run / name
                if path.exists():
                    state = json.loads(path.read_text())
                    keys = ['stage', 'current_sequence', 'sequence', 'mode', 'frames_complete', 'frames_expected', 'updated_at']
                    print(name, json.dumps({key: state[key] for key in keys if key in state}, ensure_ascii=False))
            print('Log:', log_path)
            time.sleep(15)
        if process.returncode:
            raise RuntimeError(log_path.read_text()[-6000:])
    print('전체 추론과 평가 완료:', workspace / 'results' / full_run)
else:
    print('방법 B 생략 (RUN_FULL_INFERENCE = False)')

## 8. (선택) 추적 시각화
- 실행 조건: 전체 데이터셋(영상 프레임)이 `data/MOT20`에 있을 때, 방법 B 실행 후 사용
- 생성 항목: IoU 0.1의 MOTA·오류 개수·IDF1 그래프 4개, 시퀀스별 추적·궤적 화면
- 저장 위치: `workspace/results/<실행 이름>/visual_evidence/`

In [ ]:
if RUN_FULL_INFERENCE:
    subprocess.run([sys.executable, str(pipeline / 'visualize_evaluation.py'), '--workspace', str(workspace), '--data-root', str(data_root), '--run-name', full_run], check=True)
    visuals = workspace / 'results' / full_run / 'visual_evidence'
    display(Image(filename=str(visuals / '01-mota.png')))
else:
    print('방법 B 미실행: 시각화 생략')